In [1]:
# === CELL 1: Imports and Config ===
"""
Data Download Notebook

This notebook:
1. Downloads raw kline data from Binance Vision
2. Validates and cleanses the data
3. Persists to parquet format

Outputs:
- /data/raw_data/{symbol}/*.zip
- /data/cleansed_data/{symbol}/1m.parquet
- /data/cleansed_data/{symbol}/metadata.json
"""
import os
import sys
from pathlib import Path
import zipfile
import requests

import pandas as pd
import numpy as np
from tqdm import tqdm

# Robust project root discovery
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
from utils import load_config, config_hash, ts_to_datetime, infer_unix_timestamp_unit


In [2]:
# === CELL 2: Load Configuration ===
config = load_config(ROOT / "config" / "download.yaml")
SYMBOL = config["symbol"]
START_DATE = pd.Timestamp(config["start_date"])
END_DATE = pd.Timestamp(config["end_date"])
RAW_PATH = ROOT / config["paths"]["raw_data"] / SYMBOL
CLEANSED_PATH = ROOT / config["paths"]["cleansed_data"] / SYMBOL

RAW_PATH.mkdir(parents=True, exist_ok=True)
CLEANSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Symbol: {SYMBOL}")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")


Symbol: BTCUSDT
Date range: 2023-01-01 to 2025-12-31


In [3]:
# === CELL 3: Generate Download URLs ===
def generate_month_range(start: pd.Timestamp, end: pd.Timestamp):
    """Generate (year, month) tuples for date range."""
    months = []
    current = start.replace(day=1)
    while current <= end:
        months.append((current.year, current.month))
        current += pd.DateOffset(months=1)
    return months

months = generate_month_range(START_DATE, END_DATE)
print(f"Months to download: {len(months)}")


Months to download: 36


In [4]:
# === CELL 4: Download Function ===
KLINE_COLUMNS = [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_volume", "trades",
    "taker_buy_base", "taker_buy_quote", "ignore"
]

def download_month(year: int, month: int) -> pd.DataFrame:
    """Download and parse one month of kline data."""
    url = config["url_template"].format(
        symbol=SYMBOL, year=year, month=month
    )

    zip_path = RAW_PATH / f"{SYMBOL}-1m-{year}-{month:02d}.zip"
    if not zip_path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        zip_path.write_bytes(response.content)

    with zipfile.ZipFile(zip_path) as zf:
        csv_name = zf.namelist()[0]
        with zf.open(csv_name) as f:
            df = pd.read_csv(f, header=None, names=KLINE_COLUMNS)

    return df


In [5]:
# === CELL 5: Download All Data ===
dfs = []
for year, month in tqdm(months, desc="Downloading"):
    try:
        df = download_month(year, month)
        dfs.append(df)
    except Exception as e:
        print(f"Failed {year}-{month:02d}: {e}")

if not dfs:
    raise RuntimeError("No data downloaded. Check the date range and symbol.")

raw_df = pd.concat(dfs, ignore_index=True)
print(f"Raw rows: {len(raw_df):,}")


Downloading: 100%|██████████| 36/36 [00:02<00:00, 15.80it/s]

Raw rows: 1,578,160


In [6]:
# === CELL 6: Validation and Cleansing ===
raw_df = raw_df.drop(columns=["ignore"])

dtype_map = {
    "open_time": "int64",
    "close_time": "int64",
    "open": "float64",
    "high": "float64",
    "low": "float64",
    "close": "float64",
    "volume": "float64",
    "quote_volume": "float64",
    "trades": "int64",
    "taker_buy_base": "float64",
    "taker_buy_quote": "float64",
}
raw_df = raw_df.astype(dtype_map)

STEP_MS = 60_000
DURATION_TOL_MS = 2_000
RANGE_SLACK_MS = STEP_MS * 5

start_dt = START_DATE.tz_localize("UTC") if START_DATE.tzinfo is None else START_DATE.tz_convert("UTC")
end_dt = END_DATE.tz_localize("UTC") if END_DATE.tzinfo is None else END_DATE.tz_convert("UTC")
end_dt = end_dt + pd.Timedelta(days=1) - pd.Timedelta(minutes=1)
start_ms = int(start_dt.value // 1_000_000)
end_ms = int(end_dt.value // 1_000_000)
open_min = start_ms - RANGE_SLACK_MS
open_max = end_ms + RANGE_SLACK_MS
close_min = open_min
close_max = open_max + STEP_MS

duration_ms = raw_df["close_time"] - raw_df["open_time"]
duration_ms_ok = duration_ms.between(STEP_MS - DURATION_TOL_MS, STEP_MS + DURATION_TOL_MS)
open_in_range = raw_df["open_time"].between(open_min, open_max)
close_in_range = raw_df["close_time"].between(close_min, close_max)

open_us = (raw_df["open_time"] // 1_000).astype("int64")
close_us = (raw_df["close_time"] // 1_000).astype("int64")
duration_us = close_us - open_us
duration_us_ok = duration_us.between(STEP_MS - DURATION_TOL_MS, STEP_MS + DURATION_TOL_MS)
open_us_in_range = open_us.between(open_min, open_max)
close_us_in_range = close_us.between(close_min, close_max)

convert_us = (~open_in_range | ~close_in_range | ~duration_ms_ok) & duration_us_ok & open_us_in_range & close_us_in_range
if convert_us.any():
    raw_df.loc[convert_us, "open_time"] = open_us[convert_us]
    raw_df.loc[convert_us, "close_time"] = close_us[convert_us]
print(f"Timestamp unit fixes (us->ms): {int(convert_us.sum()):,}")

raw_unit = "mixed(ms/us)" if convert_us.any() else infer_unix_timestamp_unit(int(raw_df["open_time"].median()))
normalized_unit = "ms"

raw_df = raw_df.sort_values("open_time").reset_index(drop=True)

max_gap_ms = config["max_gap_minutes"] * STEP_MS
spike_fixes = 0
for _ in range(3):
    prev_open = raw_df["open_time"].shift(1)
    next_open = raw_df["open_time"].shift(-1)
    delta_prev = raw_df["open_time"] - prev_open
    delta_next = next_open - raw_df["open_time"]
    spike = (delta_prev > max_gap_ms) & (delta_next > max_gap_ms)
    candidate_prev = prev_open + STEP_MS
    candidate_next = next_open - STEP_MS
    consistent = (candidate_prev - candidate_next).abs() <= STEP_MS
    spike = spike & consistent
    if not spike.any():
        break
    raw_df.loc[spike, "open_time"] = candidate_prev[spike].astype("int64")
    raw_df.loc[spike, "close_time"] = (raw_df.loc[spike, "open_time"] + (STEP_MS - 1)).astype("int64")
    spike_fixes += int(spike.sum())
print(f"Timestamp spike fixes: {spike_fixes:,}")

n_before = len(raw_df)
raw_df = raw_df.drop_duplicates(subset=["open_time"], keep="first")
n_dups = n_before - len(raw_df)
print(f"Duplicates removed: {n_dups:,}")


Timestamp unit fixes (us->ms): 525,600
Timestamp spike fixes: 0
Duplicates removed: 0


In [7]:
# === CELL 7: Gap Analysis ===
raw_df["expected_next"] = raw_df["open_time"] + 60_000
raw_df["actual_next"] = raw_df["open_time"].shift(-1)
raw_df["gap_minutes"] = (raw_df["actual_next"] - raw_df["expected_next"]) / 60_000

gaps = raw_df[raw_df["gap_minutes"] > 0].copy()
print(f"Total gaps: {len(gaps)}")
print(f"Max gap: {gaps['gap_minutes'].max():.0f} minutes")
major_gaps = gaps[gaps["gap_minutes"] >= config["max_gap_minutes"]]
print(f"Major gaps (>= {config['max_gap_minutes']} min): {len(major_gaps)}")

gap_summary = gaps.groupby(
    pd.cut(gaps["gap_minutes"], bins=[0, 5, 60, 1440, float("inf")])
).size()
print("Gap distribution:")
print(gap_summary)

delta_ms = raw_df["open_time"].diff()
is_gap = (delta_ms != 60_000) & delta_ms.notna()
raw_df["segment_id"] = is_gap.cumsum().astype("int64")

raw_df = raw_df.drop(columns=["expected_next", "actual_next", "gap_minutes"])


Total gaps: 1
Max gap: 80 minutes
Major gaps (>= 60 min): 1
Gap distribution:
gap_minutes
(0.0, 5.0]        0
(5.0, 60.0]       0
(60.0, 1440.0]    1
(1440.0, inf]     0
dtype: int64


C:\Users\vitil\AppData\Local\Temp\ipykernel_8932\3933719940.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gap_summary = gaps.groupby(


In [8]:
# === CELL 8: Save Cleansed Data ===
output_path = CLEANSED_PATH / "1m.parquet"
raw_df.to_parquet(output_path, index=False, engine="pyarrow")

metadata = {
    "symbol": SYMBOL,
    "start_time": int(raw_df["open_time"].min()),
    "end_time": int(raw_df["open_time"].max()),
    "row_count": len(raw_df),
    "n_timestamp_unit_fixes": int(convert_us.sum()),
    "n_timestamp_spike_fixes": int(spike_fixes),
    "raw_timestamp_unit": raw_unit,
    "normalized_timestamp_unit": normalized_unit,
    "n_gaps": int(len(gaps)),
    "n_major_gaps": int(len(major_gaps)),
    "max_gap_minutes": float(gaps["gap_minutes"].max()) if len(gaps) else 0.0,
    "config_hash": config_hash(config),
}
pd.Series(metadata).to_json(CLEANSED_PATH / "metadata.json")

print(f"Saved to: {output_path}")
print(f"Rows: {len(raw_df):,}")
print(f"Time range: {ts_to_datetime(metadata['start_time'])} to {ts_to_datetime(metadata['end_time'])}")


Saved to: C:\Users\vitil\OneDrive\Desktop\online_barrier_classifier\data\cleansed_data\BTCUSDT\1m.parquet
Rows: 1,578,160
Time range: 2023-01-01 00:00:00+00:00 to 2025-12-31 23:59:00+00:00


In [9]:
# === CELL 9: Summary Statistics ===
print("=== Data Summary ===")
print(raw_df.describe())


=== Data Summary ===
          open_time          open          high           low         close  \
count  1.578160e+06  1.578160e+06  1.578160e+06  1.578160e+06  1.578160e+06   
mean   1.719880e+12  6.544661e+04  6.546731e+04  6.542572e+04  6.544666e+04   
std    2.733512e+10  3.181839e+04  3.182729e+04  3.180945e+04  3.181837e+04   
min    1.672531e+12  1.650604e+04  1.650873e+04  1.649901e+04  1.650587e+04   
25%    1.696208e+12  3.028968e+04  3.029300e+04  3.028548e+04  3.028968e+04   
50%    1.719881e+12  6.421212e+04  6.423496e+04  6.418846e+04  6.421213e+04   
75%    1.743553e+12  9.547174e+04  9.550563e+04  9.543835e+04  9.547170e+04   
max    1.767226e+12  1.261145e+05  1.261996e+05  1.260719e+05  1.261145e+05   

             volume    close_time  quote_volume        trades  taker_buy_base  \
count  1.578160e+06  1.578160e+06  1.578160e+06  1.578160e+06    1.578160e+06   
mean   3.627450e+01  1.719880e+12  1.607775e+06  2.100876e+03    1.793496e+01   
std    8.780837e+01  2.7